# Classification Metrics & Confusion Matrix

**CLASSIFICATION_BINDER_AUDIT_V1**

This notebook demonstrates classification evaluation on a **held-out, stratified test set**.

It separates two kinds of quantities that are often mixed together:

- **Threshold-dependent metrics** — confusion matrix, accuracy, balanced accuracy, precision, recall/sensitivity, specificity, F1, MCC and Cohen's kappa.
- **Ranking metrics** — ROC AUC, Average Precision (AP), and trapezoidal area under the precision-recall curve.

Important interpretation points:

1. There is **no universal "good" cutoff** for most metrics. Compare with a relevant baseline, uncertainty, deployment costs and domain requirements.
2. Accuracy can be misleading with class imbalance.
3. Precision and negative predictive value depend strongly on prevalence.
4. A confusion matrix is tied to a chosen decision threshold.
5. **Average Precision (AP) is not the same calculation as trapezoidal PR AUC.** scikit-learn's `average_precision_score` computes a recall-weighted mean of precision values without trapezoidal interpolation.
6. Evaluate final model performance on data that were not used to fit or tune the model.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    accuracy_score,
    auc,
    average_precision_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

## 1. Create an imbalanced binary-classification problem

The positive class is deliberately less common so that the difference between plain accuracy and class-sensitive metrics is visible.

The split uses `stratify=y` so the class proportions remain similar in the training and test sets.

In [ ]:
X, y = make_classification(
    n_samples=1500,
    n_features=8,
    n_informative=5,
    n_redundant=1,
    n_repeated=0,
    n_classes=2,
    weights=[0.88, 0.12],
    class_sep=1.1,
    flip_y=0.02,
    random_state=RANDOM_STATE,
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE,
)

model = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
model.fit(X_train, y_train)

y_score = model.predict_proba(X_test)[:, 1]
threshold = 0.50
y_pred = (y_score >= threshold).astype(int)

print(f"Training samples: {len(y_train)}")
print(f"Test samples:     {len(y_test)}")
print(f"Positive prevalence in test set: {y_test.mean():.3f}")
print(f"Decision threshold: {threshold:.2f}")

## 2. Compute threshold-dependent and ranking metrics

For a binary confusion matrix ordered as labels `[0, 1]`:

\[
\begin{bmatrix}
TN & FP \\
FN & TP
\end{bmatrix}
\]

Specificity is \(TN/(TN+FP)\), while recall/sensitivity is \(TP/(TP+FN)\).

Balanced accuracy is the mean of the class recalls; for binary classification it is equivalent to \((\text{sensitivity}+\text{specificity})/2\).

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

specificity = tn / (tn + fp) if (tn + fp) else np.nan
prevalence = y_test.mean()

precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_score)

metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Balanced accuracy": balanced_accuracy_score(y_test, y_pred),
    "Precision (PPV)": precision_score(y_test, y_pred, zero_division=0),
    "Recall / sensitivity (TPR)": recall_score(y_test, y_pred, zero_division=0),
    "Specificity (TNR)": specificity,
    "F1 score": f1_score(y_test, y_pred, zero_division=0),
    "Matthews correlation coefficient (MCC)": matthews_corrcoef(y_test, y_pred),
    "Cohen's kappa": cohen_kappa_score(y_test, y_pred),
    "ROC AUC": roc_auc_score(y_test, y_score),
    "Average Precision (AP)": average_precision_score(y_test, y_score),
    "Trapezoidal PR AUC": auc(recall_curve, precision_curve),
    "Positive-class prevalence (no-skill PR baseline)": prevalence,
}

print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(cm)
print()

for name, value in metrics.items():
    print(f"{name:48s} {value:.4f}")

### AP versus trapezoidal PR AUC

These two summaries can be numerically close, but they are **not interchangeable definitions**:

- `average_precision_score(y_true, y_score)` computes **Average Precision (AP)** as a weighted mean of precision values using increments in recall.
- `auc(recall, precision)` computes the **trapezoidal area** under the supplied precision-recall points using linear interpolation.

For an imbalanced problem, the positive-class prevalence is a useful no-skill reference for the precision-recall curve. The practical usefulness of any score still depends on the operating region and costs of errors.

## 3. Visualise the confusion matrix

The confusion matrix is specific to the threshold chosen above. Changing the threshold changes TP, FP, FN and TN, and therefore changes threshold-dependent metrics.

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    labels=[0, 1],
    display_labels=["Negative", "Positive"],
    cmap="Blues",
    colorbar=False,
)
plt.title(f"Confusion matrix at threshold = {threshold:.2f}")
plt.tight_layout()
plt.show()

## 4. ROC curve

ROC AUC measures ranking across thresholds. It does **not** tell you whether predicted probabilities are well calibrated, nor whether a particular deployment threshold is appropriate.

In [ ]:
RocCurveDisplay.from_predictions(y_test, y_score)
plt.plot([0, 1], [0, 1], linestyle="--", label="Random ranking")
plt.title("ROC curve")
plt.legend()
plt.tight_layout()
plt.show()

## 5. Precision-recall curve

For a random ranking mechanism, the expected precision baseline is approximately the positive-class prevalence. PR performance is especially informative when the positive class is rare.

In [ ]:
PrecisionRecallDisplay.from_predictions(y_test, y_score, name="Logistic regression")
plt.axhline(
    prevalence,
    linestyle="--",
    label=f"No-skill prevalence baseline ({prevalence:.3f})",
)
plt.title("Precision-recall curve")
plt.legend()
plt.tight_layout()
plt.show()

## 6. Threshold trade-offs

There is no universally correct threshold. Lowering a threshold usually raises recall while increasing false positives; raising it usually does the opposite.

Choose an operating threshold from the real costs of false positives and false negatives, capacity constraints, and the deployment population.

In [ ]:
def threshold_summary(y_true, scores, threshold_value):
    pred = (scores >= threshold_value).astype(int)
    cm_local = confusion_matrix(y_true, pred, labels=[0, 1])
    tn_, fp_, fn_, tp_ = cm_local.ravel()

    specificity_ = tn_ / (tn_ + fp_) if (tn_ + fp_) else np.nan

    return {
        "threshold": threshold_value,
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "specificity": specificity_,
        "f1": f1_score(y_true, pred, zero_division=0),
    }

rows = [threshold_summary(y_test, y_score, t) for t in (0.30, 0.50, 0.70)]

header = (
    f"{'threshold':>10} {'accuracy':>10} {'precision':>10} "
    f"{'recall':>10} {'specificity':>12} {'f1':>10}"
)
print(header)
print("-" * len(header))

for row in rows:
    print(
        f"{row['threshold']:10.2f} "
        f"{row['accuracy']:10.3f} "
        f"{row['precision']:10.3f} "
        f"{row['recall']:10.3f} "
        f"{row['specificity']:12.3f} "
        f"{row['f1']:10.3f}"
    )

## Takeaways

- Report the confusion matrix or class-specific rates when the operating threshold matters.
- Do not rely on accuracy alone for imbalanced problems.
- Use ROC AUC and precision-recall summaries for **ranking**, but inspect the operating region relevant to the application.
- Keep **Average Precision (AP)** and **trapezoidal PR AUC** correctly labelled.
- Compare metrics against appropriate baselines and report uncertainty when making real decisions.
- Metric values describe predictive behaviour; they do not establish causality.